# Station (b) — the same graphs, from Python

**Everything below runs inside this browser tab.** The Python is
[Pyodide](https://pyodide.org) (CPython compiled to WebAssembly), the notebook is
[JupyterLite](https://jupyterlite.readthedocs.io), and the client is
[`rete-graph`](https://pypi.org/project/rete-graph/) from PyPI. There is no
kernel on a server, no database, and nothing to install.

The graphs are the same immutable `.rete` files the
[SPARQL console](../../playground/index.html) reads — plain objects on Cloudflare R2. The
client fetches only the byte ranges each query touches, so a 2 GB graph opens in
a tab and answers questions in megabytes.

Run the cells top to bottom (`Shift+Enter`). The first one boots Pyodide and
downloads the wheel, so give it a moment.

> Demonstration accompanying *`.rete`: Browser-Native SPARQL over Static,
> Range-Addressable RDF Files* — ISWC 2026 Posters & Demos. [Demonstration home](../../index.html).

In [ ]:
%pip install rete-graph pandas

import sys
import pandas as pd
import rete_graph as rete

print(f"rete-graph {rete.__version__} on {sys.platform!r}")

## A · Open a 2.09 GB graph without downloading it

`zenodo-records.rete` is every published Zenodo record — **7.76M records,
215,396,999 triples, 2.09 GB**, CC-BY-4.0. Opening it reads the header, the
dictionary directory and the index directories: enough to route later queries,
and a rounding error against the file.

`stats()` is the honest counter — cumulative bytes and HTTP range requests
since the graph was opened.

In [ ]:
ZENODO = "https://data.graphplaza.com/zenodo-records/zenodo-records.rete"

g = rete.open(ZENODO)
s = g.stats()
print(f"{g.quads:,} triples are queryable.")
print(
    f"opening cost {s['bytes']:,} bytes in {s['requests']} range requests "
    f"— {100 * s['bytes'] / s['fileLength']:.3f}% of the {s['fileLength']:,}-byte file"
)


def cost(graph, label, before):
    """What did the last step physically read?"""
    after = graph.stats()
    print(
        f"{label}: {after['bytes'] - before['bytes']:,} bytes in "
        f"{after['requests'] - before['requests']} range requests "
        f"(total so far {after['bytes']:,} of {after['fileLength']:,})"
    )
    return after

## B · Summary-safe — answered without opening the index

Every `.rete` carries a **dataset card**: a small section, written at build time
*from the graph itself*, holding the licence, provenance, counts, vocabularies
and exact per-predicate and per-class histograms. It sits before the dictionary,
so reading it never touches a triple index.

The histogram is **exact, not sampled** — the assertion the next cell checks.

In [ ]:
before = g.stats()
card = g.card()
before = cost(g, "the whole card", before)

predicates = pd.DataFrame(card["predicates"], columns=["predicate", "statements"])
print(f"{card['title']}\nlicence: {card['license']}\n")
print(
    f"{len(predicates)} predicates summing to {predicates.statements.sum():,} "
    f"— the file declares {card['triple_count']:,}: "
    f"{'EXACT' if predicates.statements.sum() == card['triple_count'] else 'MISMATCH'}"
)
predicates.head(12)

## C · Selective — one record out of 7.76 million

Now a query that *does* need the index. The subject is bound, so the engine
resolves it through the dictionary, picks the permutation whose prefix is bound
(SPO), and fetches only the tiles covering that range.

Watch the cost line: still a fraction of a percent of the file.

In [ ]:
rows = g.query_df(
    """
    SELECT ?p ?o WHERE {
      <https://doi.org/10.5281/zenodo.8435696> ?p ?o .
    }
    """
)
before = cost(g, f"one record ({len(rows)} triples)", before)
rows

## C′ · A join, still selective

Zenodo mints a fresh DOI for every version of a deposit, all sharing one
*concept* DOI. `dcite:isVersionOf` links them, so this returns a deposit's whole
release history — a join whose starting point is bound, and therefore still
routed rather than scanned.

In [ ]:
versions = g.query_df(
    """
    PREFIX dcite: <https://w3id.org/rete/datacite#>
    SELECT ?version ?title WHERE {
      ?version dcite:isVersionOf <https://doi.org/10.5281/zenodo.597466> .
      OPTIONAL { ?version dcite:title ?title }
    } LIMIT 50
    """
)
before = cost(g, f"version chain ({len(versions)} rows)", before)
versions.head(10)

## D · A graph somebody else published

[Open Pulse](https://openpulse.epfl.ch) is the EPFL/SDSC research-software
graph — 40,574 entities, 3,697,829 triples in 49 MB, built from the
[open-pulse ontology](https://github.com/sdsc-ordes/open-pulse-ontology) by a
third party who does not run a SPARQL endpoint.

Which EPFL labs produce the most-starred software? One file, one query, no
server.

In [ ]:
p = rete.open("https://data.graphplaza.com/open-pulse/open-pulse.rete")
start = p.stats()

labs = p.query_df(
    """
    PREFIX schema: <http://schema.org/>
    PREFIX pulse: <https://open-pulse.epfl.ch/ontology#>
    PREFIX org: <http://www.w3.org/ns/org#>
    SELECT ?org (SAMPLE(?ln) AS ?lab) (COUNT(DISTINCT ?repo) AS ?repos)
           (SUM(?st) AS ?totalStars) WHERE {
      ?repo a schema:SoftwareSourceCode ; pulse:ownedBy ?org .
      ?org org:unitOf <https://ror.org/02s376052> .
      OPTIONAL { ?org schema:name ?ln }
      OPTIONAL { ?repo pulse:githubRepoStars ?st }
    } GROUP BY ?org ORDER BY DESC(?totalStars) LIMIT 15
    """
)
cost(p, "EPFL labs by total stars", start)
labs

## The file documents itself

The card also ships **runnable starter queries**, so a graph you have never seen
tells you how to ask it something. This table comes from *inside* the file.

In [ ]:
examples = p.examples()
display(pd.DataFrame(examples)[["tier", "title", "question"]].head(10))

# ... and run one exactly as shipped:
print(examples[0]["title"], "—", examples[0]["question"])
p.query_df(examples[0]["sparql"])

## Try it on your own

`rete.open()` takes any `.rete` URL whose host sends CORS and honours HTTP
`Range` — edit the cells above and re-run them.

**One limit, stated plainly:** the third graph of this demonstration,
`crossref.rete` (3.78 B triples, 60.2 GB), needs a client newer than the
`rete-graph` release currently on PyPI. Its dictionary section passes 4 GiB,
which the 32-bit section offsets in wasm builds up to 0.2.3 could not address.
The fix ships in the next release; until then, query Crossref from the
[SPARQL console](../../playground/index.html) or the
[MCP extension](../../agent/index.html), which both carry the newer engine.

**Where next:** [demo home](../../index.html) · [SPARQL console](../../playground/index.html) ·
[agent extension](../../agent/index.html) · [Python API](https://caviri.github.io/rete/python.html) ·
[the format](https://caviri.github.io/rete/architecture.html)

Source & issues: **<https://github.com/caviri/rete>**

© 2026 Carlos Vivar Ríos — [Apache-2.0](https://github.com/caviri/rete/blob/main/LICENSE).
Datasets keep their own licences (Zenodo metadata: CC-BY-4.0; Open Pulse: GitHub
public metadata, ontology © SDSC-ORDES).